# **Proyecto final: Transformers**

In [18]:
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, AutoModelForCausalLM, AutoModelForSeq2SeqLM
import torch
import numpy as np

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Parte 1: Modelo para detectar emociones

In [4]:
# Cargar dataset
ds = load_dataset("Estwld/empathetic_dialogues_llm")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/5.22M [00:00<?, ?B/s]

data/valid-00000-of-00001.parquet:   0%|          | 0.00/806k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/798k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/19533 [00:00<?, ? examples/s]

Generating valid split:   0%|          | 0/2770 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2547 [00:00<?, ? examples/s]

In [5]:
# Extraemos las etiquetas únicas (emociones)
unique_emotions = sorted(list(set(ds['train']['emotion'])))
emotion2id = {emotion: i for i, emotion in enumerate(unique_emotions)}
id2emotion = {i: emotion for emotion, i in emotion2id.items()}

def preprocess_function(examples):
    # Tokenizamos la situación
    result = tokenizer(examples["situation"], truncation=True, padding="max_length", max_length=64)
    # Convertimos el nombre de la emoción a un ID numérico
    result["label"] = [emotion2id[emotion] for emotion in examples["emotion"]]
    return result


In [6]:
# Tokenización
model_name = "distilbert-base-uncased" # modelo utilizado
tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenized_ds = ds.map(preprocess_function, batched=True)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/19533 [00:00<?, ? examples/s]

Map:   0%|          | 0/2770 [00:00<?, ? examples/s]

Map:   0%|          | 0/2547 [00:00<?, ? examples/s]

In [7]:
# Configurar el modelo
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(unique_emotions),
    id2label=id2emotion,
    label2id=emotion2id)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [8]:
# Argumentos para el entrenamiento
training_args = TrainingArguments(
    output_dir="./empathy_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,)

In [9]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["valid"], # Usamos el set de validación que ya viene en el dataset
)

In [10]:
# Entrenamiento
trainer.train()

# Para probar
def predecir_emocion(texto):
    inputs = tokenizer(texto, return_tensors="pt", truncation=True, padding=True).to(model.device)
    with torch.no_grad():
        logits = model(**inputs).logits
    predicted_class_id = logits.argmax().item()
    return model.config.id2label[predicted_class_id]

Epoch,Training Loss,Validation Loss
1,2.021886,1.718178
2,1.450721,1.513756
3,1.160742,1.497337


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


In [11]:
# Guardar modelo entrenado
model.save_pretrained("/content/emotion_detection_model")

# Guardar tokenizer
tokenizer.save_pretrained("/content/emotion_detection_model")

print("Modelo guardado :)")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modelo guardado :)


In [12]:
import os
os.listdir("/content/emotion_detection_model")

['tokenizer.json', 'model.safetensors', 'config.json', 'tokenizer_config.json']

In [13]:
# Cargar modelo

model_path = "/content/emotion_detection_model"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

print("Modelo cargado correctamente :)")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Modelo cargado correctamente :)


In [16]:
def probal_modelo_interactivo():
    print("\n" + "="*30)
    print("Emotion classification")
    print("Write your phrase:")
    print("Enter 'exit' or 'quit' to finish the program.")
    print("="*30 + "\n")

    model.eval()

    while True:
        frase = input("Your phrase -> ")

        if frase.lower() in ["salir", "exit", "quit"]:
            print("Bye :)")
            break

        if not frase.strip():
            continue

        # Realizar la predicción
        emocion = predecir_emocion(frase)

        print(f"Emotion detected: {emocion.upper()}")
        print("-" * 20)

probal_modelo_interactivo()


Emotion classification
Write your phrase:
Enter 'exit' or 'quit' to finish the program.

Your phrase -> I have to go out into the woods at night
Emotion detected: AFRAID
--------------------
Your phrase -> exit
Bye :)


### Parte 2: Generación de respuesta

In [20]:
# Modelo generativo pre-entrenado
generator_model_name = "google/flan-t5-base"

generator_tokenizer = AutoTokenizer.from_pretrained(generator_model_name)

generator_model = AutoModelForSeq2SeqLM.from_pretrained(generator_model_name)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [21]:
# Generación de respuesta

def generar_respuesta(user_text):

    # Detectar la emoción usando el primer modelo
    emotion = predecir_emocion(user_text)

    # Crear el prompt para el segundo modelo
    prompt = f"""
You are an empathetic chatbot.

The user emotion is: {emotion}

User message:
{user_text}

Generate a short empathetic response.
"""

    # Tokenizar el input

    inputs = generator_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=128)

    # Generar respuesta
    outputs = generator_model.generate(
        **inputs,
        max_length=60,
        do_sample=True,
        temperature=0.8,
        top_p=0.9,
        top_k=50)

    # Decodificar el texto generado
    response = generator_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True)

    return emotion, response

### Chat-bot

In [22]:
# Chatbot final
def empathetic_chatbot():

    print("="*40)
    print("Empathetic Chatbot")
    print("Write 'exit' to finish")
    print("="*40)

    while True:

        user_input = input("\nYou -> ")

        if user_input.lower() in ["exit", "quit"]:
            print("\nBye :)")
            break

        if not user_input.strip():
            continue

        # Generar respuesta
        emotion, response = generar_respuesta(user_input)

        print(f"\nDetected emotion: {emotion}")
        print(f"Bot -> {response}")

In [ ]:
empathetic_chatbot()